In [2]:
import keras
import numpy as np
from keras import layers
from keras.utils import to_categorical
import shap
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [3]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


In [4]:
num_classes = 10
input_shape = (28, 28, 1)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)


In [5]:
model = keras.Sequential([
    layers.Input(shape=input_shape),
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(x_train, y_train, batch_size=128, epochs=3, validation_split=0.1)

# 解決問題：初始化模型輸入屬性
_ = model.predict(x_train[:1])


Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.9157 - loss: 0.2740 - val_accuracy: 0.9822 - val_loss: 0.0631
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9736 - loss: 0.0836 - val_accuracy: 0.9868 - val_loss: 0.0476
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9805 - loss: 0.0626 - val_accuracy: 0.9878 - val_loss: 0.0389
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step


In [6]:
def generate_fgsm_samples(model, x_data, y_data, epsilon=0.1):
    x_adv = x_data.copy()
    y_true = tf.convert_to_tensor(y_data, dtype=tf.float32)
    x_tensor = tf.convert_to_tensor(x_data, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        predictions = model(x_tensor)
        loss = tf.keras.losses.categorical_crossentropy(y_true, predictions)

    gradients = tape.gradient(loss, x_tensor)
    x_adv += epsilon * tf.sign(gradients)
    x_adv = tf.clip_by_value(x_adv, 0, 1)
    return x_adv.numpy()

x_train_adv = generate_fgsm_samples(model, x_train, y_train, epsilon=0.1)


In [8]:
logit_layer_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[-2].output)
background = x_train[np.random.choice(x_train.shape[0], 20, replace=False)]
explainer = shap.DeepExplainer(logit_layer_model, background)

def compute_shap_values(explainer, data, batch_size=10):
    shap_values = []
    for i in range(0, len(data), batch_size):
        shap_values.extend(explainer.shap_values(data[i:i + batch_size]))
    return np.array(shap_values)

shap_values_original = compute_shap_values(explainer, x_train[:1000])
print(shap_values_original.shape)
shap_values_adv = compute_shap_values(explainer, x_train_adv[:1000])


(1000, 28, 28, 1, 128)


In [9]:
def extract_shap_signature(shap_values):
    return np.array([sv.flatten() for sv in shap_values])

shap_signature_original = extract_shap_signature(shap_values_original)
print(shap_signature_original.shape)
shap_signature_adv = extract_shap_signature(shap_values_adv)


(1000, 100352)


In [ ]:
X = np.vstack([shap_signature_original, shap_signature_adv])
y = np.array([0] * len(shap_signature_original) + [1] * len(shap_signature_adv))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

detector_model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(2, activation="softmax"),
])

detector_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
detector_model.fit(X_train, y_train, batch_size=32, epochs=10, validation_split=0.1)


In [ ]:
y_pred = np.argmax(detector_model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))
